We first create a suitable python enviroment to run inDelphi
```bash
conda create -n legacy_env python=3.6 pandas=0.23.4 scikit-learn=0.20.0 scipy=1.1.0 numpy=1.15.3 ipykernel -c defaults

conda activate legacy_env

python -m ipykernel install --user --name legacy_env --display-name "Python 3.6 (inDelphi)"

```

In [1]:
import sys
sys.path.append('C:\\Users\\tzion\\Documents\\GitHub\\inDelphi-model') # CHANGE TO PATH USED

import os
import glob
import inDelphi
import pandas as pd
import warnings
import time
import multiprocessing as mp
from functools import partial
# 1. Suppress Pandas/Scikit-learn FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

print(sys.platform)

win32


Since we are using Windows, need to fix the .pkl files - remove this if using linux

In [2]:
def fix_pickle_line_endings(directory):
    # The hex for \r\n is 0D 0A. We want to convert it back to \n (0A).
    WINDOWS_NEWLINE = b'\r\n'
    UNIX_NEWLINE = b'\n'
    
    count = 0
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.pkl'):
                file_path = os.path.join(root, file)
                
                with open(file_path, 'rb') as f:
                    content = f.read()
                
                # Check if the problematic Windows byte is present
                if WINDOWS_NEWLINE in content:
                    print(f"Fixing: {file}")
                    
                    # Create a backup just in case
                    with open(file_path + '.bak', 'wb') as bak:
                        bak.write(content)
                    
                    # Replace \r\n with \n
                    new_content = content.replace(WINDOWS_NEWLINE, UNIX_NEWLINE)
                    
                    with open(file_path, 'wb') as f:
                        f.write(new_content)
                    count += 1
                    
    print(f"Finished. Fixed {count} files.")


model_base_path = r'C:\Users\tzion\Documents\GitHub\inDelphi-model'
fix_pickle_line_endings(model_base_path)

Finished. Fixed 0 files.


We want to predict using all diff models

In [3]:
# Define the paths
data_dir = '../data/'
output_dir = './inDelphi_preds/'
CELL_TYPES = ['mESC', 'U2OS', 'HEK293', 'HCT116', 'K562']

# Ensure output directory exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

Some progress bar

In [4]:



# 1. Suppress warnings and define silence context
warnings.simplefilter(action='ignore', category=FutureWarning)

class SilenceStdout:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

# 2. Advanced Progress Bar with ETA
def update_progress(current, total, start_time, prefix=""):
    bar_length = 25
    fraction = current / total
    elapsed_time = time.time() - start_time
    
    # Calculate ETA
    if current > 0:
        avg_time_per_item = elapsed_time / current
        remaining_items = total - current
        eta_seconds = int(avg_time_per_item * remaining_items)
        
        # Format time to MM:SS
        eta_str = f"{eta_seconds // 60:02d}:{eta_seconds % 60:02d}"
    else:
        eta_str = "--:--"

    arrow = int(fraction * bar_length) * '#'
    padding = (bar_length - len(arrow)) * ' '
    
    sys.stdout.write(f"\r{prefix} [{arrow}{padding}] {int(fraction*100)}% ({current}/{total}) ETA: {eta_str} ")
    sys.stdout.flush()

We first build a set of all sequences and their PAM position

In [5]:
print("Scanning datasets for unique sequences...")
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

# Remove LINDEL files if present since they contain N
csv_files = [f for f in csv_files if 'Lindel' not in f]

# TEMP only keep SPROUT files
#csv_files = [f for f in csv_files if 'SPROUT' in f]

unique_pairs = set()

for f in csv_files:
    temp_df = pd.read_csv(f)
    for _, row in temp_df.iterrows():
        try:
            # We store as a tuple (seq, cutsite)
            unique_pairs.add((row['sequence'], int(row['PAM position']) - 3))
        except:
            continue

unique_list = list(unique_pairs)
total_unique = len(unique_list)
print(f"Found {total_unique} unique sequence/cutsite combinations.")

Scanning datasets for unique sequences...
Found 131139 unique sequence/cutsite combinations.


We then predict on those sequences (using multiprocessing since inDelphi uses only 1 core)

In [6]:
import time
import sys

# Cache structure: {cell_type: {(seq, cutsite): fs_freq}}
prediction_cache = {ct: {} for ct in CELL_TYPES}

for cell_type in CELL_TYPES:
    print(f"\nProcessing {cell_type}:")
    
    # Force model re-initialization for the new cell type
    inDelphi.init_flag = False 
    with SilenceStdout():
        inDelphi.init_model(celltype=cell_type)
        
    start_time = time.time()
    
    for i, (seq, cutsite) in enumerate(unique_list):
        current_count = i + 1
        
        try:
            # We call the core predict logic directly
            with SilenceStdout():
                _, stats = inDelphi.predict(seq, cutsite)
            val = stats['Frameshift frequency'] / 100
        except Exception:
            val = None
            
        # Store in cache
        prediction_cache[cell_type][(seq, cutsite)] = val
        
        # UPDATE PROGRESS 
        update_progress(current_count, total_unique, start_time, prefix=f"  {cell_type: <7}")
        
    print() 


Processing mESC:
  mESC    [#########################] 100% (131139/131139) ETA: 00:00 

Processing U2OS:
  U2OS    [#########################] 100% (131139/131139) ETA: 00:00 

Processing HEK293:
  HEK293  [#########################] 100% (131139/131139) ETA: 00:00 

Processing HCT116:
  HCT116  [#########################] 100% (131139/131139) ETA: 00:00 

Processing K562:
  K562    [#########################] 100% (131139/131139) ETA: 00:00 


We then merge the results back to the .csv files and save

In [7]:
print("\nMapping results back to files...")
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    
    for cell_type in CELL_TYPES:
        # Create the column by looking up the (seq, cutsite) tuple in our cache
        df[f'inDelphi_FS_{cell_type}'] = df.apply(
            lambda row: prediction_cache[cell_type].get((row['sequence'], int(row['PAM position']) - 3)), 
            axis=1
        )
    
    output_path = os.path.join(output_dir, file_name)
    df.to_csv(output_path, index=False)
    print(f"Saved: {file_name}")

print("\nAll tasks completed successfully.")


Mapping results back to files...
Saved: ALDIT_HAP1.csv
Saved: ALDIT_Jurkat.csv
Saved: ALDIT_Jurkat_DNTTKO.csv
Saved: ALDIT_K562.csv
Saved: ALDIT_K562_DNTTOE.csv
Saved: FORECasT_BOB.csv
Saved: FORECasT_CHO.csv
Saved: FORECasT_HAP1.csv
Saved: FORECasT_K562.csv
Saved: FORECasT_K562_2A_TREX2.csv
Saved: FORECasT_K562_eCAS9.csv
Saved: FORECasT_K562_TREX2.csv
Saved: FORECasT_mESC.csv
Saved: FORECasT_RPE1.csv
Saved: SPROUT_T.csv
Saved: SPROUT_T_CROTON_VERSION.csv
Saved: XCRISP_FORECasT_HAP1.csv
Saved: XCRISP_FORECasT_mESC.csv
Saved: XCRISP_FORECasT_TREX2.csv
Saved: XCRISP_inDelphi_mESC.csv
Saved: XCRISP_inDelphi_mESC_NHEJdeficient.csv
Saved: XCRISP_inDelphi_U2OS.csv

All tasks completed successfully.
